# 03 — Classical Baselines

Trains and evaluates **SVM**, **Random Forest**, and **XGBoost** with:
- Hand-crafted window-level features (mean, std, min, max, RMS, FFT energy)
- LOSO (leave-one-subject-out) cross-validation
- SHAP feature importance for tree models

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from src.config import PROCESSED_DIR, METRICS_DIR
from src.baselines import extract_features, run_baselines_loso

sns.set_theme(style='whitegrid')
print('Ready.')

## 1. Load Processed Data

In [ ]:
try:
    X_p2   = np.load(f'{PROCESSED_DIR}/pamap2_X.npy')
    y_p2   = np.load(f'{PROCESSED_DIR}/pamap2_y.npy')
    subj_p2 = np.load(f'{PROCESSED_DIR}/pamap2_subjects.npy')
    print(f'PAMAP2: X={X_p2.shape}, y={y_p2.shape}')
    DATA_LOADED = True
except FileNotFoundError:
    print('Processed data not found — run 02_preprocessing_pipeline.ipynb first.')
    DATA_LOADED = False

## 2. Feature Extraction

In [ ]:
if DATA_LOADED:
    X_feat = extract_features(X_p2)
    print(f'Feature matrix: {X_feat.shape}  (n_windows, n_features)')
    print(f'Features per window = {X_feat.shape[1]}')
    
    # Quick sanity check
    df_feat = pd.DataFrame(X_feat[:, :6], columns=['mean_0','mean_1','mean_2','std_0','std_1','std_2'])
    df_feat.describe()

## 3. Run LOSO Baselines

In [ ]:
if DATA_LOADED:
    results = run_baselines_loso(X_p2, y_p2, subj_p2, dataset_name='pamap2')

## 4. Results Comparison

In [ ]:
# Load saved results (works even after re-running)
results_file = Path(METRICS_DIR) / 'pamap2_baselines.json'
if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)

    # Build comparison table
    rows = []
    for model, m in results.items():
        rows.append({
            'Model': model,
            'Mean Accuracy': f"{m['mean_accuracy']:.4f}",
            'Std Accuracy':  f"±{m['std_accuracy']:.4f}",
            'Mean Macro-F1': f"{m['mean_macro_f1']:.4f}",
            'Std Macro-F1':  f"±{m['std_macro_f1']:.4f}",
        })
    df_res = pd.DataFrame(rows).set_index('Model')
    print(df_res.to_string())

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    models = list(results.keys())
    accs = [results[m]['mean_accuracy'] for m in models]
    errs = [results[m]['std_accuracy']  for m in models]
    ax.bar(models, accs, yerr=errs, capsize=5, color=['steelblue','coral','mediumseagreen'][:len(models)])
    ax.set_ylim(0, 1)
    ax.set_ylabel('LOSO Accuracy')
    ax.set_title('Classical Baseline Comparison — PAMAP2')
    plt.tight_layout()
    plt.show()
else:
    print('No results file yet — run baselines first.')

## 5. SHAP Feature Importance (Random Forest)

In [ ]:
if DATA_LOADED:
    import shap
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from src.config import SEED

    # Train RF on full data for SHAP demo
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_feat)
    rf.fit(X_scaled, y_p2)

    # SHAP
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_scaled[:200])  # use subset for speed

    n_channels = X_p2.shape[2]
    feature_names = (
        [f'mean_ch{i}'   for i in range(n_channels)] +
        [f'std_ch{i}'    for i in range(n_channels)] +
        [f'min_ch{i}'    for i in range(n_channels)] +
        [f'max_ch{i}'    for i in range(n_channels)] +
        [f'rms_ch{i}'    for i in range(n_channels)] +
        [f'fft_ch{i}'    for i in range(n_channels)]
    )

    shap.summary_plot(
        shap_values if isinstance(shap_values, np.ndarray) else np.abs(np.array(shap_values)).mean(0),
        X_scaled[:200],
        feature_names=feature_names,
        max_display=20,
        plot_type='bar',
        show=True,
    )
else:
    print('Data not loaded.')